# DoorDash Delivery ETA — Preparação Controlada dos Dados

Este notebook executa e valida o pipeline reproduzível de preparação controlada definido em `src/data_preparation.py`, gerando três visões dos dados:
- `df_raw`: leitura fiel do arquivo original (`data/raw/historical_data.csv`), sem alterações.
- `df_analysis`: cópia com tipos corrigidos, variável-alvo, indicadores de validade e atributos iniciais, sem exclusão de registros e preservando valores negativos.
- `df_model`: cópia preparada para modelagem inicial, removendo apenas alvos ausentes/não positivos e convertendo valores negativos em `NaN` nas 4 colunas especificadas (sem imputação, sem clipping e sem remoção de outliers plausíveis).

In [ ]:
import sys
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_preparation import prepare_datasets, NEGATIVE_TO_NAN_COLS

RAW_PATH = PROJECT_ROOT / "data" / "raw" / "historical_data.csv"
df_raw, df_analysis, df_model, summary = prepare_datasets(RAW_PATH)

print("Dimensões:")
print(f"  df_raw:      {summary.raw_shape}")
print(f"  df_analysis: {summary.analysis_shape}")
print(f"  df_model:    {summary.model_shape}")
print(f"Registros removidos: {summary.removed_rows} -> {summary.removed_reasons}")

## 1. Verificação de Tipos (`df_model`)

In [ ]:
display(df_model.dtypes.to_frame("dtype"))

## 2. Contabilização de Alvos e Indicadores de Validade (`df_analysis`)

In [ ]:
target_summary_df = pd.DataFrame([
    {"métrica": k, "quantidade": v, "percentual_%": round(v / summary.target_counts["total_rows"] * 100, 4)}
    for k, v in summary.target_counts.items()
])
display(target_summary_df)

## 3. Valores Negativos Convertidos em `NaN` (`df_model`)

In [ ]:
neg_summary = pd.DataFrame([
    {
        "coluna": col,
        "negativos_df_analysis": int((df_analysis[col] < 0).sum()),
        "convertidos_para_nan_df_model": summary.negatives_converted_to_nan[col],
        "negativos_restantes_df_model": int((df_model[col] < 0).sum()),
    }
    for col in NEGATIVE_TO_NAN_COLS
])
display(neg_summary)

## 4. Ausências por Coluna (`df_raw` vs `df_analysis` vs `df_model`)

In [ ]:
missing_cmp = pd.DataFrame({
    "raw_na": df_raw.isna().sum(),
    "analysis_na": df_analysis.isna().sum(),
    "model_na": df_model.isna().sum(),
}).fillna(0).astype(int)
missing_cmp["model_na_%"] = (missing_cmp["model_na"] / len(df_model) * 100).round(4)
display(missing_cmp[missing_cmp.sum(axis=1) > 0])

## 5. Atributos Iniciais Criados e Comparação da Variável-Alvo

In [ ]:
print("Indicador busy_greater_than_onshift (df_model):")
display(df_model["busy_greater_than_onshift"].value_counts(dropna=False).to_frame("contagem"))

print("Resumo de dasher_count_difference (df_model):")
display(df_model["dasher_count_difference"].describe().to_frame("dasher_count_difference"))

print("Comparação da distribuição da variável-alvo (segundos e minutos):")
target_cmp = pd.DataFrame({
    "analysis_sec_valid": df_analysis["delivery_duration_seconds"].dropna().describe(),
    "model_sec": df_model["delivery_duration_seconds"].describe(),
    "analysis_min_valid": df_analysis["delivery_duration_minutes"].dropna().describe(),
    "model_min": df_model["delivery_duration_minutes"].describe(),
})
display(target_cmp)